# AURA — training the reuse model

This notebook takes cache traces and produces the `model_bundle.json` artifacts that the
Rust engine loads. It is the same pipeline as `python -m aura_train.cli all`, unrolled so
each step is visible and the intermediate tables can be read.

Run it top to bottom. Nothing here requires a Rust toolchain, a GPU, or any data you do
not already have — step 2 falls back to a synthetic trace generator if you have nothing
else, so the notebook never dead-ends.

**What comes out at the end**

* `reuse_gbdt_h10s.json`, `reuse_gbdt_h60s.json`, `reuse_gbdt_h600s.json` — the gradient
  boosted reuse models, one per prediction horizon
* `reuse_linear_h60s.json` — the online logistic model the engine uses during cold start
* a per-regime metrics table, a calibration plot, a gain-importance plot, and a
  counterfactual cost comparison
* optionally, all of the above pushed to Supabase and marked active

**What this notebook will not do:** train on a random split. Every number reported here
is on regimes the model has never seen. See step 3.

## 0. Install

LightGBM is the model we ship. If the install fails (it does, occasionally, on a fresh
Colab image), the trainer falls back to scikit-learn's `HistGradientBoostingClassifier`
and exports the *identical* bundle format — the Rust tree walker cannot tell the two
apart. So a failed LightGBM install is a slower model, not a broken notebook.

In [ ]:
%pip install -q lightgbm pandas pyarrow scikit-learn matplotlib onnxmltools skl2onnx onnxruntime supabase

import importlib

for module in ["lightgbm", "pyarrow", "skl2onnx", "onnxmltools", "onnxruntime", "supabase"]:
    try:
        importlib.import_module(module)
        print(f"  {module:14s} ok")
    except ImportError as exc:
        print(f"  {module:14s} MISSING ({exc}) - the pipeline has a fallback for this")

## 1. Get the training package

The feature builder is not reimplemented in this notebook, on purpose. It has to stay
byte-identical to the one the Rust engine runs, and the only way to guarantee that is to
import the single copy that the golden-vector test pins.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Point this at your checkout. Any of these work:
#   * a repo already present in the Colab VM  (e.g. after mounting Drive)
#   * a git URL, cloned below
REPO_URL = "https://github.com/VH26-KalaDhua/aura.git"
LOCAL_CANDIDATES = [Path("/content/aura/training"), Path("aura/training"), Path("training"), Path(".")]

training_dir = next((p for p in LOCAL_CANDIDATES if (p / "aura_train").is_dir()), None)

if training_dir is None:
    print(f"cloning {REPO_URL}")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/aura"], check=True)
    training_dir = Path("/content/aura/training")

training_dir = training_dir.resolve()
if str(training_dir) not in sys.path:
    sys.path.insert(0, str(training_dir))

import aura_train
from aura_train.config import load_config
from aura_train.features import FEATURE_GROUPS, FEATURE_NAMES

print(f"aura_train {aura_train.__version__} from {training_dir}")
print(f"{len(FEATURE_NAMES)} features: {', '.join(FEATURE_NAMES)}")

## 2. Get some traces

Three ways in, in descending order of how much the resulting model is worth:

1. **Upload traces you generated with the Rust binary** (`aura-bench --emit-trace`).
   These are the only traces that carry a measured cost vector per object, which is what
   the economic half of the model learns from.
2. **Download them from Supabase Storage**, if a previous run archived them there.
3. **Generate synthetic ones.** Pure Python, no Rust, seven regimes. Weaker than the real
   simulator, but it exercises every code path and produces a model you can inspect.

Public research traces (Twitter, Wikipedia CDN, libCacheSim `oracleGeneral`, …) are also
supported by `aura_train.traces` — see the README. They have no cost metadata, so they
train the reuse head only.

Set `TRACE_SOURCE` and run the cell.

In [ ]:
import os
import shutil

TRACE_SOURCE = "synthetic"   # "upload" | "supabase" | "synthetic"

cfg = load_config(
    trace_dir=Path("/content/data/traces"),
    dataset_dir=Path("/content/data/dataset"),
    model_dir=Path("/content/models"),
    report_dir=Path("/content/reports"),
)
cfg.ensure_dirs()

if TRACE_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()          # pick one or more *.csv.gz (and their *.meta.json)
    for name in uploaded:
        shutil.move(name, cfg.trace_dir / name)

elif TRACE_SOURCE == "supabase":
    from aura_train.supabase_io import Session

    session = Session()
    rows = session.select("aura_traces", order="created_at", limit=20)
    if not rows:
        raise SystemExit("no traces registered in aura_traces; use 'synthetic' instead")
    for row in rows:
        bucket, _, key = str(row["storage_path"]).partition("/")
        payload = session.storage_download(bucket, key)
        (cfg.trace_dir / str(row["name"])).write_bytes(payload)
        print(f"  {row['name']}  {row['rows']} rows")

elif TRACE_SOURCE == "synthetic":
    from aura_train.synthetic import generate_trace_set

    generate_trace_set(cfg.trace_dir, requests_per_regime=60_000, unique_keys=4_000, seed=42)

else:
    raise ValueError(f"unknown TRACE_SOURCE {TRACE_SOURCE!r}")

from aura_train.traces import TraceMeta, detect_format, discover_traces

for path in discover_traces(cfg.trace_dir):
    meta = TraceMeta.from_path(path)
    print(f"{path.name:34s} {detect_format(path):14s} "
          f"{path.stat().st_size / 1e6:7.1f} MB  scenario={meta.scenario}")

## 3. Build the dataset

One forward pass builds the 16 features (never looking beyond the current timestamp), one
reverse pass builds the labels (`was this key accessed again within 10 s / 60 s / 600 s?`).
The two passes share no state, which is what keeps the features honest.

Two things to look at in the output:

* **Censored rows dropped.** Near the end of a trace we cannot observe whether a key came
  back, so a zero there means "unknown", not "no". Those rows are removed. Keeping them
  teaches the model that late traffic is worthless.
* **The class-balance table.** At the 10 s horizon most accesses are *not* reused; at
  600 s most are. That is why three horizons exist — the engine needs a different question
  answered for an admission decision than for an eviction decision.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s")
logging.getLogger("matplotlib").setLevel(logging.WARNING)

from aura_train.dataset import build_dataset, class_balance_table, load_dataset

report = build_dataset(cfg, progress=True)

print(f"\nrows {report.rows:,}   shards {len(report.shards)}   "
      f"censored dropped {report.censored_dropped:,}\n")
print("rows by split")
for split, count in sorted(report.rows_by_split.items()):
    print(f"  {split:8s} {count:>10,d}")
print("\nrows by regime")
for regime, count in sorted(report.rows_by_regime.items()):
    print(f"  {regime:24s} {count:>10,d}")
print("\nclass balance")
print(class_balance_table(report).to_string(index=False))

frame = load_dataset(cfg)
frame.head()

### Why the split is not random

A random split puts two accesses to the same key, seconds apart, on opposite sides of the
boundary. The model memorises the key and the AUC looks superb until it meets production.

So the split is by **regime**:

| split | regimes |
|---|---|
| train | `steady`, `zipf_shift_moderate`, `analytics_stable` — first 80% by time |
| val | the same regimes, trailing 20% by time (early stopping and calibration only) |
| test | `flash_crowd`, `scan`, `expensive_tail`, `cost_spike` — never seen in any form |

Every number below step 4 is on workload shapes the model was never trained on.

In [ ]:
print(cfg.split.train_regimes, "->", "train/val")
print(cfg.split.test_regimes, "->", "test")
frame.groupby(["split", "regime"]).size().unstack(fill_value=0)

## 4. Train

Three horizons, early stopping against the held-out validation *segments* rather than a
random internal fraction, then the ablation loop.

The ablations are the part worth reading. Each variant drops one feature group and
retrains from scratch. If dropping the cost block does not move the test AUC, the cost
block is decoration and should be deleted rather than defended. Reporting this honestly
is cheaper than discovering it later.

In [ ]:
from aura_train.train_gbdt import ABLATIONS, run_ablations, train_gbdt
from aura_train.train_linear import train_linear

gbdt_models = [train_gbdt(cfg, frame, horizon) for horizon in cfg.horizons_ms]
linear_model = train_linear(cfg, frame, cfg.primary_horizon_ms)

primary = next(m for m in gbdt_models if m.horizon_ms == cfg.primary_horizon_ms)

print(f"\n{'model':20s} {'backend':14s} {'trees':>6s} {'val auc':>8s} {'val pr':>8s} "
      f"{'logloss':>8s} {'sec':>6s}")
for model in gbdt_models:
    print(f"reuse_gbdt_{cfg.horizon_label(model.horizon_ms):9s} {model.backend:14s} "
          f"{model.best_iteration:6d} {model.metrics['auc']:8.4f} {model.metrics['pr_auc']:8.4f} "
          f"{model.metrics['logloss']:8.4f} {model.train_seconds:6.1f}")
print(f"{'reuse_linear':20s} {'sgd':14s} {'-':>6s} {linear_model.metrics['auc']:8.4f} "
      f"{linear_model.metrics['pr_auc']:8.4f} {linear_model.metrics['logloss']:8.4f}")

In [ ]:
ablation_table = run_ablations(cfg, frame, cfg.primary_horizon_ms, ABLATIONS)
ablation_table

## 5. Evaluate

Four outputs, answering four different questions.

**Per-regime AUC** — does the model rank reuse correctly on workload shapes it has never
seen? A pooled number would average a flash crowd together with steady traffic and hide
exactly the failure we care about.

**Calibration** — is a predicted 0.7 actually a 70% chance? The engine multiplies this
probability by a dollar cost to get a value density, so a well-ranked but badly scaled
output produces correctly ordered, numerically meaningless decisions.

**Gain importance** — which features are carrying the model. Read it alongside the
ablation table: importance says what the trees used, ablation says what they could not do
without.

**Counterfactual replay** — the one that matters. Replay the held-out trace through a
cache simulator once per policy (LRU, LFU, GDSF, AURA) and charge each one the
regeneration cost of every miss it allowed. AUC does not pay the bill; this does. A model
can win on AUC and lose here by filling the cache with objects that were cheap to
regenerate.

In [ ]:
from aura_train.evaluate import evaluate

evaluation = evaluate(
    cfg,
    frame,
    primary.predict,
    importance=primary.gain_importance(),
    horizon_ms=primary.horizon_ms,
    ablations=ablation_table,
)
evaluation.per_regime

In [ ]:
from IPython.display import Image, display

for key in ["per_regime_png", "calibration_png", "importance_png", "replay_png"]:
    path = evaluation.paths.get(key)
    if path is not None:
        display(Image(filename=str(path)))

In [ ]:
replay = evaluation.replay
pivot = replay.pivot_table(index="regime", columns="policy", values="regen_cost_usd")
pivot["aura_saving_vs_lru"] = 1 - pivot["aura"] / pivot["lru"]
pivot["aura_saving_vs_gdsf"] = 1 - pivot["aura"] / pivot["gdsf"]
pivot.round(4)

### The cold-start blend

The GBDT above is useless in the first minute of a deployment: it has to be trained,
exported and loaded. Three predictors cover that gap and the engine blends them by
confidence — heuristic, then the online logistic model, then the GBDT. A predictor whose
confidence is below `[engine] ml_confidence_floor` (0.20) contributes nothing at all,
which is what makes loading a new bundle safe.

The cell below shows what the cache would predict at each stage, on the same rows.

In [ ]:
import numpy as np

from aura_train.train_linear import blend, cold_start_model, heuristic_probability

test = frame[(frame["split"] == "test") & (frame[cfg.censored_column(cfg.primary_horizon_ms)] == 0)]
sample = test.sample(min(20_000, len(test)), random_state=42)
x = sample[list(FEATURE_NAMES)].to_numpy(dtype=float)
y = sample[cfg.label_column(cfg.primary_horizon_ms)].to_numpy()

from sklearn.metrics import roc_auc_score

stages = {
    "heuristic (no data at all)": heuristic_probability(x),
    "cold-start prior (shipped in the binary)": cold_start_model().predict(x),
    "fitted linear (a few thousand requests in)": linear_model.predict(x),
    "gbdt (bundle loaded)": primary.predict(x),
    "blend at 0.5 gbdt confidence": blend(
        heuristic_probability(x), linear_model.predict(x), primary.predict(x), 0.8, 0.5
    ),
}
for name, probability in stages.items():
    print(f"{name:46s} auc {roc_auc_score(y, probability):.4f}")

## 6. Export

The bundle is the only artifact the Rust engine ever sees. Before anything is written,
`export.py` scores the exported trees through a reference walker — a line-for-line
description of what `aura-core` does — and compares it against the trainer's own
prediction over 1000 held-out rows. A disagreement above `1e-6` fails the export.

That check is what catches an off-by-one in a leaf index or a flipped comparison. Without
it, a tree-encoding bug produces a bundle that loads cleanly and serves wrong numbers
forever.

In [ ]:
from aura_train.export import export_all, read_bundle, sample_rows, verify_onnx

parity_rows = sample_rows(test[list(FEATURE_NAMES)].to_numpy(dtype=float), 1000, cfg.seed)
artifacts = export_all(cfg.model_dir, gbdt_models, linear_model, parity_rows)

for artifact in artifacts:
    print(f"{artifact.name:22s} parity delta {artifact.parity_delta:.3e}  {artifact.bundle_path}")
    if artifact.onnx_path is not None:
        try:
            delta = verify_onnx(artifact.onnx_path, parity_rows, primary.predict(parity_rows))
            print(f"{'':22s} onnx  delta {delta:.3e}  {artifact.onnx_path}")
        except Exception as exc:
            print(f"{'':22s} onnx  check skipped: {exc}")

bundle = read_bundle(cfg.model_dir / "reuse_gbdt_h60s.json")
print(f"\n{bundle['name']} v{bundle['version']}  kind={bundle['kind']} "
      f"trees={len(bundle['trees'])}  metrics={bundle['metrics']}")

In [ ]:
from aura_train.export import bundle_schema_errors

for path in sorted(cfg.model_dir.glob("reuse_*.json")):
    errors = bundle_schema_errors(read_bundle(path))
    print(f"{path.name:26s} {'ok' if not errors else errors}")

## 7. Push to Supabase

Credentials come from Colab secrets (the key icon in the left sidebar) or from plain
environment variables. Add two secrets and enable notebook access for both:

* `SUPABASE_URL`
* `SUPABASE_SERVICE_ROLE_SECRET_KEY`

The service role key bypasses row-level security, so it belongs in the secrets panel and
nowhere else — never in a cell, never in a commit.

The cell uploads each bundle to the `aura-models` bucket (creating it if needed), inserts
the `aura_models` row, and flips `is_active`. The schema has a partial unique index that
makes two active versions of the same model name impossible, so the flip is atomic from
the engine's point of view.

In [ ]:
import os

try:
    from google.colab import userdata

    for key in ["SUPABASE_URL", "SUPABASE_SERVICE_ROLE_SECRET_KEY"]:
        try:
            os.environ[key] = userdata.get(key)
        except Exception as exc:
            print(f"{key} not available from Colab secrets ({exc}); falling back to os.environ")
except ImportError:
    pass   # not running in Colab; SUPABASE_* are expected in the environment already

ACTIVATE = True

from aura_train.supabase_io import Session, register_model, set_active, upload_bundle

session = Session()
for path in sorted(cfg.model_dir.glob("reuse_*.json")):
    bundle = read_bundle(path)
    onnx = path.with_suffix(".onnx")
    storage_path, onnx_storage = upload_bundle(
        path, onnx if onnx.exists() else None, bucket=cfg.storage_bucket, session=session
    )
    register_model(path, storage_path, onnx_storage, is_active=ACTIVATE, session=session)
    if ACTIVATE:
        set_active(str(bundle["name"]), str(bundle["version"]), session=session)
    print(f"pushed {bundle['name']:22s} {bundle['version']}  -> {storage_path}")

In [ ]:
from aura_train.supabase_io import list_models

import pandas as pd

rows = list_models(limit=20)
pd.DataFrame(rows)[["name", "version", "kind", "horizon_ms", "is_active", "storage_path"]]

## 8. Tell the running engine to pick it up

The server does not poll. It loads the active bundle on boot and otherwise only when it is
asked, so a model rollout is an explicit, observable action rather than something that
happens silently between two requests.

Run the command printed below against your server. `GET /v1/explain/{key}` afterwards will
show `"predictor": "gbdt"` and the new `predictor_confidence`; confidence starts low and
climbs as the engine observes how well the new bundle's predictions match reality, so the
new model does not take over the cache the instant it lands.

In [ ]:
AURA_SERVER = "http://localhost:8080"

print("reload the model:")
print(f"  curl -X POST {AURA_SERVER}/v1/model/reload \\")
print("       -H 'Content-Type: application/json' \\")
print("       -d '{\"source\":\"supabase\"}'")
print()
print("then confirm it took:")
print(f"  curl -s {AURA_SERVER}/v1/explain/recent?limit=1 | python -m json.tool")
print(f"  curl -s {AURA_SERVER}/healthz")